# PHILDER — Replicação

Notebook principal da replicação do PHILDER (BISPO et al., 2025). Executa, em ordem, as quatro etapas da metodologia do artigo — consolidação da base (A), processamento do texto (B), balanceamento, treino e teste (C) e avaliação de desempenho (D) — usando o pacote `philder`. Para várias rodadas ou vários modelos, use `rodar_lote.py`.

## Preparação

Importa os módulos do pacote `philder` e cria a estrutura de diretórios do projeto.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import glob
import os

from philder import infra
from philder import database_consolidation as consolidacao
from philder import data_processing as processamento
from philder import database_balance as balanceamento
from philder import performance_assessment as avaliacao

# --- Preparação ---
infra.criar_estrutura_diretorios()

## Configuração do experimento

Define o modelo a treinar, o alvo do *undersampling* (502 legítimos e 501 phishing) e as pastas de saída. O número da rodada é gerado automaticamente.

In [ ]:
# --- Configuração do experimento (modelo + padrão de saída) ---
# Padrão de saída:
#   modelo/<modelo_escolhido>/               -> modelo final treinado
#   modelo/<modelo_escolhido>/checkpoints/   -> output_dir do Trainer
#   resultados/<modelo_escolhido>/resultado_final-<rodada>_<ddmmaaaa>.csv
#   resultados/<modelo_escolhido>/curva_perda-<rodada>_<ddmmaaaa>.png
# O número da rodada é calculado automaticamente (maior rodada do dia na pasta + 1).
modelos_artigo = {
    "tinybert":   "huawei-noah/TinyBERT_General_4L_312D",
    "minilm":     "microsoft/MiniLM-L12-H384-uncased",
    "distilbert": "distilbert-base-uncased",
    "mobilebert": "google/mobilebert-uncased",
    "albert":     "albert-base-v2",
}
modelo_escolhido = "tinybert"            # <-- único ponto a trocar entre experimentos
checkpoint_modelo = modelos_artigo[modelo_escolhido]

# Alvo do undersampling por classe (0 = legítimo, 1 = phishing)
# {0: 502, 1: 501} -> fiel ao artigo (Figura 2: 1.003 registros)
# None             -> todas as classes no tamanho da minoritária (501 + 501)
undersampling_alvo = {0: 502, 1: 501}
data_execucao = datetime.date.today().strftime("%d%m%Y")

pasta_modelo = os.path.join("modelo", modelo_escolhido)
pasta_checkpoints = os.path.join(pasta_modelo, "checkpoints")
pasta_resultados = os.path.join("resultados", modelo_escolhido)


def proxima_rodada(pasta: str, data: str) -> int:
    """Próximo número de rodada do dia em `pasta` (1 se ainda não houver nenhuma)."""
    arquivos = glob.glob(os.path.join(pasta, f"resultado_final-*_{data}.csv"))
    numeros = [int(os.path.basename(a).split("-")[1].split("_")[0]) for a in arquivos]
    return max(numeros, default=0) + 1


print(f"Modelo: {modelo_escolhido} ({checkpoint_modelo}) | saída: {pasta_modelo}/ e {pasta_resultados}/ "
      f"| undersampling: {undersampling_alvo}")

# A) DATABASE CONSOLIDATION

Importa os e-mails legítimos (SpamAssassin) e de phishing (corpus Nazario), remove duplicados pelo texto processado, amostra a quantidade de e-mails do artigo e cria o rótulo de cada mensagem.

In [ ]:
# --- Módulo A: consolidação da base ---
brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")

# Deduplica pelo texto processado e amostra a quantidade de e-mails do artigo
legitimos_amostrados, phishing_amostrados = consolidacao.amostrar_quantidade_artigo(
    brutos_legitimos, brutos_phishing, deduplicar_antes=True
)

df = consolidacao.consolidar_dataset(legitimos_amostrados, phishing_amostrados)

# B) DATA PROCESSING

Limpa o corpo dos e-mails: remove o cabeçalho SMTP, substitui links por `URL` e endereços de e-mail por `EMAILADDRESS`, normaliza o texto e remove *stopwords*.

In [ ]:
# --- Módulo B: processamento de texto ---
df = processamento.pipeline_processamento(df)

# C) DATABASE BALANCE

Aplica o *undersampling*, divide os dados em treino e teste (70/30, estratificado), tokeniza, treina o modelo por 10 épocas, mede o tempo de teste e salva o modelo treinado.

In [ ]:
# --- Módulo C: balanceamento + treino + teste ---
df_balanceado = balanceamento.aplicar_undersampling(df, n_por_classe=undersampling_alvo)
df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)

tokenizer, modelo_pre_treinado = balanceamento.carregar_tokenizer_e_modelo(checkpoint_modelo)
dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
trainer, tempo_treino = balanceamento.treinar_modelo(
    modelo_pre_treinado, tokenizer, dataset_treino, dataset_teste, output_dir=pasta_checkpoints
)
predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
balanceamento.salvar_modelo(trainer, tokenizer, caminho=pasta_modelo)

### Verificação dos dados

Calcula as impressões digitais MD5 dos conjuntos balanceado, de treino e de teste, para confirmar que todas as execuções usam os mesmos dados, e registra as versões do ambiente.

In [ ]:
import hashlib, glob, os, sys
import torch, transformers

def impressao(df):
    texto = "\n".join(sorted(df["body_processado"].astype(str)))
    return hashlib.md5(texto.encode("utf-8")).hexdigest()

print("balanceado:", impressao(df_balanceado))
print("treino:    ", impressao(df_treino))
print("teste:     ", impressao(df_teste))

arquivos = glob.glob(os.path.join("dados_philder", "spamassassin", "easy_ham", "*"))
print("3 primeiros arquivos do glob:", [os.path.basename(a) for a in arquivos[:3]])
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)

# D) PERFORMANCE ASSESSMENT

Calcula as métricas de classificação (acurácia, precisão, *recall* e F1) e de eficiência (TES, IES e RTDE), gera a curva de perda e salva os resultados da rodada.

In [ ]:
# --- Módulo D: avaliação de desempenho ---
metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
# Separa o tempo de avaliação por época do tempo de treino (TES bruto x TES líquido)
tempo_aval = avaliacao.calcular_tempo_avaliacao(trainer.state.log_history)
eficiencia = avaliacao.calcular_metricas_eficiencia(
    metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste), tempo_avaliacao=tempo_aval
)
# Saída por modelo, com número de rodada automático
rodada = proxima_rodada(pasta_resultados, data_execucao)
avaliacao.gerar_curva_perda(
    trainer.state.log_history,
    caminho_imagem=os.path.join(pasta_resultados, f"curva_perda-{rodada}_{data_execucao}.png"),
    titulo=f"Curva de perda — fine-tuning {modelo_escolhido} (rodada {rodada})",
)
avaliacao.salvar_resultados(
    {"modelo": modelo_escolhido, "checkpoint": checkpoint_modelo, "rodada": rodada, **metricas, **eficiencia,
     "undersampling": str(undersampling_alvo)},
    caminho_csv=os.path.join(pasta_resultados, f"resultado_final-{rodada}_{data_execucao}.csv"),
)

# ALBERT-SPECIFIC CONFIGURATION TEST

Experimento complementar e independente: treina o ALBERT com os hiperparâmetros recomendados no seu artigo original (taxa de aprendizado 1e-5, lote 32 e aquecimento de 6%), sem alterar o pacote `philder`.

In [ ]:
# ===== EXPERIMENTO COMPLEMENTAR — ALBERT com os hiperparâmetros do artigo original =====
# Fonte: Lan et al. (ALBERT), Tabela 14, tarefa SST-2 -> lr 1e-5, batch 32, warmup ≈ 6%.
# Célula independente: NÃO altera o pacote philder nem as células do protocolo principal.
# Muda só: learning rate, batch de treino e warmup. Mantém: 10 épocas, weight_decay 0.01,
# max_length 128, batch de avaliação 8 (tempo de teste comparável), dados 502/501, semente 42.
import os, math, time, hashlib, datetime
from transformers import Trainer, TrainingArguments
from philder import database_consolidation as consolidacao_exp
from philder import data_processing as processamento_exp
from philder import database_balance as balanceamento_exp
from philder import performance_assessment as avaliacao_exp

EXP_MODELO = "albert"
EXP_CHECKPOINT = "albert-base-v2"
EXP_LR = 1e-5
EXP_BATCH = 32
EXP_EPOCAS = 10
EXP_WARMUP_FRACAO = 0.06
EXP_UNDERSAMPLING = {0: 502, 1: 501}

# --- Dados (mesmo pipeline do lote final) + impressão digital ---
leg = consolidacao_exp.importar_corpus_spamassassin("dados_philder/spamassassin")
phi = consolidacao_exp.importar_corpus_nazario("dados_philder/Nazario.csv")
leg_am, phi_am = consolidacao_exp.amostrar_quantidade_artigo(leg, phi, deduplicar_antes=True)
df_exp = processamento_exp.pipeline_processamento(consolidacao_exp.consolidar_dataset(leg_am, phi_am))
df_bal_exp = balanceamento_exp.aplicar_undersampling(df_exp, n_por_classe=EXP_UNDERSAMPLING)
df_tr_exp, df_te_exp = balanceamento_exp.dividir_treino_teste(df_bal_exp)
_md5 = lambda d: hashlib.md5("\n".join(sorted(d["body_processado"].astype(str))).encode("utf-8")).hexdigest()
hashes_exp = {"hash_balanceado": _md5(df_bal_exp), "hash_treino": _md5(df_tr_exp), "hash_teste": _md5(df_te_exp)}
print(hashes_exp)
# Confere se o conjunto de teste é o mesmo do lote oficial (impressão digital MD5 de referência).
assert hashes_exp["hash_teste"] == "ccfa3c3612628cdfcee16eab995a7d00", "Teste diferente da referência 502/501 — parar e investigar."

# --- Modelo, tokenização e hiperparâmetros ---
tok_exp, mod_exp = balanceamento_exp.carregar_tokenizer_e_modelo(EXP_CHECKPOINT)
ds_tr_exp, ds_te_exp = balanceamento_exp.tokenizar_dataset(df_tr_exp, df_te_exp, tok_exp)
passos_totais = math.ceil(len(ds_tr_exp) / EXP_BATCH) * EXP_EPOCAS
passos_warmup = round(EXP_WARMUP_FRACAO * passos_totais)
print(f"Passos de treino: {passos_totais} | warmup: {passos_warmup} passos ({EXP_WARMUP_FRACAO:.0%})")

pasta_exp = os.path.join("resultados", "experimento_hp", EXP_MODELO)
os.makedirs(pasta_exp, exist_ok=True)
data_exp = datetime.date.today().strftime("%d%m%Y")
rodada_exp = 1 + max([int(f.split("-")[1].split("_")[0]) for f in os.listdir(pasta_exp)
                      if f.startswith("resultado_final-")] or [0])

args_exp = TrainingArguments(
    output_dir=os.path.join("modelo", "experimento_hp", EXP_MODELO, "checkpoints"),
    num_train_epochs=EXP_EPOCAS,
    learning_rate=EXP_LR,
    per_device_train_batch_size=EXP_BATCH,
    per_device_eval_batch_size=8,
    warmup_steps=passos_warmup,
    eval_strategy="epoch",
    save_strategy="no",
    weight_decay=0.01,
    logging_steps=2,          # 23 passos por época com batch 32: registra a perda com mais detalhe
    dataloader_pin_memory=False,
    report_to="none",
)
trainer_exp = Trainer(model=mod_exp, args=args_exp, train_dataset=ds_tr_exp, eval_dataset=ds_te_exp,
                      compute_metrics=balanceamento_exp._metricas_durante_treino)

inicio = time.time()
trainer_exp.train()
tempo_treino_exp = time.time() - inicio
predicoes_exp, tempo_teste_exp = balanceamento_exp.testar_modelo(trainer_exp, ds_te_exp)

# --- Métricas e saídas (pasta separada do protocolo principal) ---
metricas_exp = avaliacao_exp.calcular_metricas_primarias(df_te_exp["class_label"].tolist(), predicoes_exp)
tempo_aval_exp = avaliacao_exp.calcular_tempo_avaliacao(trainer_exp.state.log_history)
eficiencia_exp = avaliacao_exp.calcular_metricas_eficiencia(
    metricas_exp["accuracy"], tempo_treino_exp, tempo_teste_exp, len(df_te_exp), tempo_avaliacao=tempo_aval_exp)
avaliacao_exp.gerar_curva_perda(
    trainer_exp.state.log_history,
    caminho_imagem=os.path.join(pasta_exp, f"curva_perda-{rodada_exp}_{data_exp}.png"),
    titulo=f"Curva de perda — {EXP_MODELO} (lr {EXP_LR:g}, batch {EXP_BATCH}, warmup {EXP_WARMUP_FRACAO:.0%}) — rodada {rodada_exp}")
avaliacao_exp.salvar_resultados(
    {"modelo": EXP_MODELO, "checkpoint": EXP_CHECKPOINT, "rodada": rodada_exp,
     "experimento": "hiperparametros_artigo_original", "learning_rate": EXP_LR, "batch_treino": EXP_BATCH,
     "epocas": EXP_EPOCAS, "warmup_steps": passos_warmup, "passos_totais": passos_totais,
     **metricas_exp, **eficiencia_exp, **hashes_exp, "undersampling": str(EXP_UNDERSAMPLING)},
    caminho_csv=os.path.join(pasta_exp, f"resultado_final-{rodada_exp}_{data_exp}.csv"))
print(f"=== Concluído: accuracy={metricas_exp['accuracy']:.4f} | treino={tempo_treino_exp:.1f}s | teste={tempo_teste_exp:.2f}s ===")

# DEDUPLICATION CHECK

Mostra exemplos de e-mails brutos diferentes que se tornam idênticos após o processamento do texto, o que justifica a deduplicação pelo texto processado.

In [ ]:
# ===== Evidência da deduplicação pelo texto processado =====
# Mostra, para cada rótulo, 2 e-mails BRUTOS diferentes que viram o MESMO body_processado
# após o Módulo B — exatamente o critério usado em amostrar_quantidade_artigo(deduplicar_antes=True).
# Usa brutos_legitimos / brutos_phishing já carregados na célula do Módulo A. Apenas exibe.
import difflib
import pandas as pd

LIMITE = 600   # caracteres exibidos de cada texto

def _grupos_duplicados(emails_brutos):
    campos = [consolidacao.extrair_campos_email(m) for m in emails_brutos]
    df_tmp = processamento.pipeline_processamento(pd.DataFrame(campos))
    df_tmp["bruto"] = emails_brutos
    dup = df_tmp[df_tmp.duplicated(subset="body_processado", keep=False)]
    grupos = list(dup.groupby("body_processado", sort=False))
    # separa grupos em que os brutos são idênticos dos que só coincidem DEPOIS do processamento
    so_apos = [(t, g) for t, g in grupos if g["bruto"].nunique() > 1]
    identicos = [(t, g) for t, g in grupos if g["bruto"].nunique() == 1]
    return df_tmp, so_apos, identicos

def _mostrar(texto):
    texto = texto.replace("\r", "\n")
    return texto[:LIMITE] + (f" [...] (+{len(texto) - LIMITE} caracteres)" if len(texto) > LIMITE else "")

def _onde_diferem(a, b, contexto=60):
    a, b = a.replace("\r", "\n"), b.replace("\r", "\n")
    blocos = [op for op in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes() if op[0] != "equal"]
    for tag, i1, i2, j1, j2 in blocos[:3]:
        print(f"  • {tag}: A[{i1}:{i2}] = {a[i1:i2][:contexto]!r}  |  B[{j1}:{j2}] = {b[j1:j2][:contexto]!r}")
    if len(blocos) > 3:
        print(f"  • ... e mais {len(blocos) - 3} trecho(s) diferente(s)")

def evidenciar(emails_brutos, nome):
    df_tmp, so_apos, identicos = _grupos_duplicados(emails_brutos)
    removidos = df_tmp["body_processado"].duplicated(keep="first").sum()
    print("#" * 95)
    print(f"{nome}: {len(emails_brutos)} e-mails brutos -> {len(emails_brutos) - removidos} únicos ({removidos} descartados)")
    print(f"  grupos de duplicatas: {len(so_apos) + len(identicos)} "
          f"({len(identicos)} com texto bruto idêntico; {len(so_apos)} só iguais APÓS o processamento)")
    print("#" * 95)
    if not so_apos:
        print("Nenhum par diferente no bruto e igual após o processamento.\n")
        return
    texto_proc, g = max(so_apos, key=lambda tg: len(tg[0]))   # o par com mais conteúdo, para ficar ilustrativo
    a, b = g["bruto"].iloc[0], g["bruto"].iloc[1]
    print(f"\n--- E-mail A (bruto, posição {g.index[0]} na lista, {len(a)} caracteres) ---\n{_mostrar(a)}")
    print(f"\n--- E-mail B (bruto, posição {g.index[1]} na lista, {len(b)} caracteres) ---\n{_mostrar(b)}")
    print("\n--- Onde os brutos diferem ---")
    _onde_diferem(a, b)
    print(f"\n--- body_processado (IGUAL nos dois; {len(texto_proc)} caracteres) ---\n{_mostrar(texto_proc)}")
    print(f"\nResultado: só o 1º ({g.index[0]}) fica no pool; o(s) outro(s) {list(g.index[1:])} é(são) descartado(s).\n")

evidenciar(brutos_phishing, "PHISHING (Nazario)")
evidenciar(brutos_legitimos, "HAM / LEGÍTIMO (SpamAssassin)")

### Recursos computacionais (opcional)

Exibe o sistema operacional, o processador, a memória e as versões das bibliotecas do ambiente.

In [ ]:
# Célula opcional — levantamento dos recursos computacionais

import platform, psutil, torch, transformers

print("SO:", platform.platform())
print("CPU:", platform.processor())
print("Núcleos físicos/lógicos:", psutil.cpu_count(logical=False), "/", psutil.cpu_count())
print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)